<a href="https://colab.research.google.com/github/arantzalopezun-pixel/Proyecto-de-Estructuras-de-Datos-y-algoritmos/blob/main/Avance_proyeco_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto de Estructuras de Datos y Algoritmos



Módulo 1 (Clase Base)

In [ ]:
class Interval:  # Define la clase Interval para intervalos unidimensionales
    def __init__(self, left_x: float, right_x: float, segment_id: int = 0):  # Constructor de la clase
        if left_x > right_x:  # Checa si el extremo izquierdo es mayor al derecho
            raise ValueError("Extremo izquierdo:", {left_x} , "no puede ser mayor al derecho: ",{right_x})
        self.left_x = float(left_x)      # Guarda el límite izquierdo como float
        self.right_x = float(right_x)    # Guarda el límite derecho como float
        self.segment_id = segment_id     # Pone el ID

    def contains(self, q_x: float) -> bool:  # Método para ver si un punto está dentro del intervalo
        """Checa si el punto q_x intersecta el intervalo (stabbing query)."""
        return self.left_x <= q_x <= self.right_x #Regresa True si q_x esta entre el lim izq y derecho

    def __repr__(self):  # Método para que sea legible cuando se muestre en pantalla
        return f"Interval[{self.left_x}, {self.right_x}] (ID: {self.segment_id})" #devuelve cadena con los datos del intervalo


#pruebas
print("Prueba de la Clase Interval")
int1 = Interval(2, 8, segment_id=1)  # Instancia un primer intervalo de 2 a 8 con ID 1
int2 = Interval(10, 16, segment_id=2)  # Instancia un segundo intervalo de 10 a 16 con ID 2
print("Intervalo 1:", int1)
print("Intervalo 2:", int2)
print("¿int1 contiene el punto x = 5.0?:", int1.contains(5.0))  # Evalúa x = 5 en int1 (debe devolver True)
print("¿int1 contiene el punto x = 12.0?:", int1.contains(12.0))  # Evalúa x = 12 en int1 (debe devolver False)




Prueba de la Clase Interval
Intervalo 1: Interval[2.0, 8.0] (ID: 1)
Intervalo 2: Interval[10.0, 16.0] (ID: 2)
¿int1 contiene el punto x = 5.0?: True
¿int1 contiene el punto x = 12.0?: False


Módulo 2: Fuerza Bruta

In [ ]:
class BruteForceSearch: #Clase para buscar secuencialmente o con fuerza bruta
    """Implementación de búsqueda exhaustiva lineal O(n).""" #el tiempo

    def __init__(self, intervals: list[Interval]): #constructor que recibe la lista de objetos de Interval
        self.intervals = list(intervals) #Crea una copia de la lista de intervalos

    def query(self, q_x: float) -> list[Interval]: # Método para buscar los intervalos que contengan el punto q_x
        """Recorre secuencialmente todos los intervalos comprobando intersección."""
        result = [] # Inicia una lista vacía para almacenar los intervalos
        for interval in self.intervals: # Itera sobre cada uno de los intervalos guardados
            if interval.contains(q_x): # Llama al método contains() para verificar si el punto q_x intersecta
                result.append(interval) # Agrega el intervalo a la lista si la condición fue True
        return result # Devuelve la lista con todos los intervalos encontrados

#pruebas
print("Prueba de el método de Fuerza Bruta ")
datos_prueba = [Interval(2, 8, 1), Interval(10, 16, 2), Interval(4, 12, 3)]  # Crea una lista con 3 intervalos de prueba
buscador_bf = BruteForceSearch(datos_prueba)  # Crea el objeto de búsqueda por fuerza bruta
res_bf = buscador_bf.query(6.0)  # Consulta qué intervalos intersectan el punto x = 6.0
print("Resultado de búsqueda lineal para x = 6.0:")  # Encabezado para la salida del resultado
for elem in res_bf:  # Recorre la lista de intervalos devueltos
    print(" ->", elem)  # Imprime cada intervalo hallado (debería mostrar ID 1 e ID 3)


Prueba de el método de Fuerza Bruta 
Resultado de búsqueda lineal para x = 6.0:
 -> Interval[2.0, 8.0] (ID: 1)
 -> Interval[4.0, 12.0] (ID: 3)


Módulo 3: Interval Tree

In [ ]:
class IntervalTreeNode: # Clase que hace nodos individuales de la estructura en árbol
    """Nodo individual de un Interval Tree."""

    def __init__(self, x_mid: float): # Constructor del nodo que recibe el valor del punto medio x_mid
        self.x_mid =  x_mid # Guarda la coordenada x mediana asignada a este nodo
        self.L_left = []   # Ordenado ascendentemente por left_x
        # Inicia la lista de intervalos cruzados ordenados por el extremo izquierdo
        self.L_right = []  # Ordenado descendentemente por right_x
        # Inicializa la lista de intervalos cruzados ordenados por el extremo derecho
        self.left_child = None # Enlace hacia el subárbol izquierdo (inicialmente vacío)
        self.right_child = None # Enlace hacia el subárbol derecho (inicialmente vacío)

class IntervalTree: # Clase que administra y construye el Árbol de Intervalos
    """Estructura de Árbol de Intervalos con búsquedas O(log n + k)."""

    def __init__(self, intervals: list[Interval]): # Constructor que toma la lista inicial de intervalos
        self.root = self._build_tree(intervals) # Construye el árbol llamando al método recursivo y guarda la raíz

    def _build_tree(self, intervals: list[Interval]): # Método interno recursivo para construir la estructura del árbol
        if not intervals: # Condición base: si la lista está vacía, no hay nada que construir
            return None # Retorna None indicando la ausencia de un nodo

        # 1. Obtener todos los puntos extremos y calcular la mediana
        endpoints = [] # Crea una lista para recolectar todas las coordenadas x de los extremos
        for interval in intervals: # Itera a través de todos los intervalos
            endpoints.extend([interval.left_x, interval.right_x]) # Agrega los extremos izquierdo y derecho a la lista
        endpoints.sort() # Ordena numéricamente la lista de extremos de menor a mayor
        x_mid = endpoints[len(endpoints) // 2] # Determina la mediana tomando el punto central de la lista

        node = IntervalTreeNode(x_mid) # Instancia un nuevo nodo con la mediana calculada
        I_left, I_right, I_mid = [], [], [] # Crea tres listas para clasificar los intervalos respecto a x_mid


        # 2. Clasificar intervalos respecto a x_mid
        for interval in intervals: # Clasifica cada intervalo según su posición respecto al punto medio x_mid
            if interval.right_x < x_mid: # El intervalo está totalmente a la izquierda de x_mid
                I_left.append(interval) # Se añade al grupo de la izquierda
            elif interval.left_x > x_mid: # El intervalo está totalmente a la derecha de x_mid
                I_right.append(interval)        # Se añade al grupo de la derecha
            else:                               # El intervalo cruza o abarca la coordenada x_mid
                I_mid.append(interval) # Se añade al grupo central


        # 3. Construir listas L_left y L_right
        node.L_left = sorted(I_mid, key=lambda i: i.left_x) # Ordena los intervalos centrales de menor a mayor por left_x
        node.L_right = sorted(I_mid, key=lambda i: i.right_x, reverse=True) # Ordena los intervalos centrales de mayor a menor por right_x6t5r


        # 4. Construcción recursiva de subárboles
        node.left_child = self._build_tree(I_left) # Llama recursivamente para construir el hijo izquierdo
        node.right_child = self._build_tree(I_right) # Llama recursivamente para construir el hijo derecho

        return node # Devuelve el nodo procesado como raíz del subárbol actual

    def query(self, q_x: float) -> list[Interval]: # Método público para consultar qué intervalos intersectan el punto q_x
        """Inicia la consulta de punto desde la raíz."""
        result = [] # Inicializa la lista donde se guardarán todas las coincidencias
        self._query_node(self.root, q_x, result) # Inicia la búsqueda recursiva a partir del nodo raíz
        return result # Retorna los resultados acumulados

    def _query_node(self, node: IntervalTreeNode, q_x: float, result: list[Interval]): # Método interno recursivo de búsqueda
        if node is None: # Si la rama está vacía, no hace nada
            return

        if q_x < node.x_mid: # Si el punto q_x se ubica a la izquierda de la mediana del nodo actual
            # Recorrer L_left hasta que left_x > q_x
            for interval in node.L_left: # Revisa los intervalos centrales ordenados por su inicio
                if interval.left_x <= q_x: # Si el inicio del intervalo es menor o igual a q_x, sí intersecta
                    result.append(interval) # Agrega el intervalo a la lista de coincidencias
                else: # Si el inicio es mayor a q_x, los siguientes también lo serán
                    break
            self._query_node(node.left_child, q_x, result) # Continúa la búsqueda de forma recursiva en el hijo izquierdo
        else: # Si el punto q_x es mayor o igual a la mediana del nodo actual
            # Recorrer L_right hasta que right_x &lt; q_x
            for interval in node.L_right: # Revisa los intervalos centrales ordenados descendentemente por su fin
                if interval.right_x >= q_x: # Si el fin del intervalo es mayor o igual a q_x, sí intersecta
                    result.append(interval) # Agrega el intervalo a la lista de coincidencias
                else: # Si el fin es menor a q_x, los siguientes también lo serán
                    break
            self._query_node(node.right_child, q_x, result) # Continúa la búsqueda de forma recursiva en el hijo derecho

#pruebas
print("Prueba del Árbol de Intervalos ")
arbol_prueba = IntervalTree(datos_prueba)  # Construye el árbol usando la lista de datos del módulo anterior
print("Punto medio (x_mid) en la raíz del árbol:", arbol_prueba.root.x_mid)  # Imprime la mediana hallada en la raíz
res_arbol = arbol_prueba.query(6.0)  # Ejecuta la consulta optimizada para x = 6.0
print("Resultado de búsqueda en Árbol para x = 6.0:")  # Encabezado para imprimir resultados
for elem in res_arbol:  # Recorre la lista de intervalos retornados por el árbol
    print(" ->", elem)  # Imprime cada intervalo devuelto

Prueba del Árbol de Intervalos 
Punto medio (x_mid) en la raíz del árbol: 10.0
Resultado de búsqueda en Árbol para x = 6.0:
 -> Interval[4.0, 12.0] (ID: 3)
 -> Interval[2.0, 8.0] (ID: 1)


Módulo 4: Pruebas

In [ ]:
import unittest


class TestIntervalQueries(unittest.TestCase):  # Clase para definir las pruebas automatizadas heredando de TestCase

  def setUp(self):  # Método que prepara los objetos e información requerida antes de cada prueba
    # Crea un objeto Interval de prueba con ID 1
    self.s1 = Interval(2, 8, segment_id=1) # Pista s1 s
    # Crea un objeto Interval de prueba con ID 2
    self.s2 = Interval(10, 16, segment_id=2) # Pista s2
    # Crea un objeto Interval de prueba con ID 3
    self.s3 = Interval(4, 12, segment_id=3) # Pista s3
    self.intervals = [self.s1, self.s2, self.s3]  # Agrupa los 3 objetos en una lista

  def test_brute_force_query(self): # Método de prueba para verificar el funcionamiento de la Fuerza Bruta
    bf = BruteForceSearch(self.intervals) # Instancia la clase de búsqueda lineal
    res = bf.query(6.0) # Realiza la consulta para x = 6.0
    # En q_x = 6, deben responder s1 (2..8) y s3 (4..12)
    ids = [i.segment_id for i in res] # Extrae una lista únicamente con los IDs devueltos
    self.assertIn(1, ids)# Afirma que el ID 1 debe estar dentro de la lista de respuestas
    self.assertIn(3, ids) # Afirma que el ID 3 debe estar dentro de la lista de respuestas
    self.assertNotIn(2, ids) # Afirma que el ID 2 no debe formar parte de la respuesta

    def test_interval_tree_query(self): # Método de prueba para verificar el Árbol de Intervalos
      tree = IntervalTree(self.intervals) # Instancia la clase Árbol de Intervalos
      res = tree.query(6.0) # Realiza la consulta para x = 6.0
      ids = [i.segment_id for i in res] # Extrae la lista de IDs obtenidos del árbol
      self.assertIn(1, ids) # Afirma que el ID 1 debe estar dentro de la respuesta
      self.assertIn(3, ids) # Afirma que el ID 3 debe estar dentro de la respuesta
      self.assertNotIn(2, ids) # Afirma que el ID 2 no debe estar dentro de la respuesta

    def test_empty_query(self): # Método de prueba para consultas que no coinciden con ningún intervalo
      tree = IntervalTree(self.intervals) # Instancia la clase Árbol de Intervalos
      res = tree.query(20.0) # Ningún intervalo cubre x = 20
      self.assertEqual(len(res), 0) # Afirma que el total de elementos encontrados debe ser cero

  if __name__ == '__main__': # Comprueba si el archivo/celda se está ejecutando directamente
    unittest.main(argv=[''], exit = False) # Ejecuta las pruebas indicando argv y exit=False para ser compatible con Colab


----------------------------------------------------------------------
Ran 0 tests in 0.000s

NO TESTS RAN
